# Emotion Detection Model Training
## Fine-tuned Facial Expression Recognition (FER) for 7 Emotion Classes

This notebook trains a deep learning model to detect emotions in images and videos.
- **Emotion Classes**: Happy, Sad, Angry, Surprised, Disgusted, Fearful, Neutral
- **Approach**: Fine-tuning pre-trained models (ResNet50, EfficientNet)
- **Framework**: PyTorch with torchvision


## 1. Install Required Libraries

In [2]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install opencv-python pillow numpy pandas matplotlib seaborn scikit-learn tqdm albumentations
# !pip install mtcnn  # For face detection
# !pip install tensorboard
# print("All libraries installed successfully!")
!pip install seaborn

## 2. Import Libraries

In [4]:
import shutil

# Remove hidden checkpoint folders
shutil.rmtree("./train/.ipynb_checkpoints", ignore_errors=True)
shutil.rmtree("./test/.ipynb_checkpoints", ignore_errors=True)

print("Hidden folders removed!")

Hidden folders removed!


In [5]:
import os

print(os.listdir("./train"))
print(os.listdir("./test"))

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 3. Define Configuration

In [7]:
class Config:
    # Dataset paths - UPDATE THESE PATHS
    TRAIN_DATA_PATH = './train'  # Path to training images
    TEST_DATA_PATH = './test'    # Path to test images
    # VAL_DATA_PATH = './data/val'      # Path to validation images (optional)
    
    # Model parameters
    MODEL_NAME = 'resnet50'  # Options: 'resnet50', 'efficientnet_b3', 'resnet152'
    NUM_CLASSES = 7
    IMG_SIZE = 224
    BATCH_SIZE = 32
    NUM_EPOCHS = 4
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4
    
    # Training parameters
    NUM_WORKERS = 4
    PIN_MEMORY = True
    PATIENCE = 10  # Early stopping patience
    
    # Emotion classes
    EMOTION_CLASSES = {
        0: 'Angry',
        1: 'Disgusted',
        2: 'Fearful',
        3: 'Happy',
        4: 'Neutral',
        5: 'Sad',
        6: 'Surprised'
    }
    
    # Checkpoint paths
    CHECKPOINT_DIR = './checkpoints'
    BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
    FINAL_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'final_model.pth')
    
    def __init__(self):
        os.makedirs(self.CHECKPOINT_DIR, exist_ok=True)

config = Config()
print("Configuration loaded!")
print(f"Number of classes: {config.NUM_CLASSES}")
print(f"Classes: {config.EMOTION_CLASSES}")

Configuration loaded!
Number of classes: 7
Classes: {0: 'Angry', 1: 'Disgusted', 2: 'Fearful', 3: 'Happy', 4: 'Neutral', 5: 'Sad', 6: 'Surprised'}


## 4. Custom Dataset Class

In [8]:
class EmotionDataset(Dataset):
    """
    Custom dataset for emotion detection.
    Assumes folder structure: data/train/emotion_name/*.jpg
    """
    def __init__(self, data_path, transform=None, emotion_to_idx=None):
        self.data_path = data_path
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Create emotion to index mapping if not provided
        if emotion_to_idx is None:
            emotion_folders = sorted([d for d in os.listdir(data_path) 
                                     if os.path.isdir(os.path.join(data_path, d))])
            emotion_to_idx = {emotion: idx for idx, emotion in enumerate(emotion_folders)}
        
        self.emotion_to_idx = emotion_to_idx
        self.idx_to_emotion = {v: k for k, v in emotion_to_idx.items()}
        
        # Load image paths and labels
        for emotion, idx in emotion_to_idx.items():
            emotion_path = os.path.join(data_path, emotion)
            if os.path.exists(emotion_path):
                for img_file in os.listdir(emotion_path):
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
                        self.images.append(os.path.join(emotion_path, img_file))
                        self.labels.append(idx)
        
        print(f"Loaded {len(self.images)} images from {data_path}")
        print(f"Class distribution: {self._get_class_distribution()}")
    
    def _get_class_distribution(self):
        distribution = {}
        for label in self.labels:
            emotion = self.idx_to_emotion[label]
            distribution[emotion] = distribution.get(emotion, 0) + 1
        return distribution
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return None
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label, img_path

print("EmotionDataset class defined!")

EmotionDataset class defined!


## 5. Data Augmentation & Preprocessing

In [9]:
# Define transforms for training and testing
train_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Data augmentation transforms defined!")

Data augmentation transforms defined!


## 6. Load Datasets

In [11]:
# Load datasets
train_dataset = EmotionDataset(config.TRAIN_DATA_PATH, transform=train_transform)

# If validation path exists, use it; otherwise split training data
if os.path.exists(config.VAL_DATA_PATH):
    val_dataset = EmotionDataset(config.VAL_DATA_PATH, transform=test_transform)
else:
    print("Validation path not found. Splitting 80-20 from training data...")
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        train_dataset, [train_size, val_size]
    )

test_dataset = EmotionDataset(config.TEST_DATA_PATH, transform=test_transform)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=config.BATCH_SIZE,
#     shuffle=False,
#     num_workers=config.NUM_WORKERS,
#     pin_memory=config.PIN_MEMORY
# )

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

print(f"\nDataset Summary:")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Loaded 12067 images from ./train
Class distribution: {'angry': 1826, 'disgust': 436, 'fear': 1865, 'happy': 2285, 'neutral': 1974, 'sad': 1944, 'surprise': 1737}


AttributeError: 'Config' object has no attribute 'VAL_DATA_PATH'

## 7. Build Model Architecture

In [ ]:
class EmotionDetectionModel(nn.Module):
    """
    Fine-tuned model for emotion detection.
    Uses pre-trained backbone with custom classification head.
    """
    def __init__(self, model_name='resnet50', num_classes=7, pretrained=True):
        super(EmotionDetectionModel, self).__init__()
        
        if model_name == 'resnet50':
            self.backbone = models.resnet50(pretrained=pretrained)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()  # Remove final layer
        
        elif model_name == 'resnet152':
            self.backbone = models.resnet152(pretrained=pretrained)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        
        elif model_name == 'efficientnet_b3':
            self.backbone = models.efficientnet_b3(pretrained=pretrained)
            num_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()
        
        else:
            raise ValueError(f"Model {model_name} not supported")
        
        # Custom classification head
        self.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.head(features)
        return logits

# Create model
model = EmotionDetectionModel(
    model_name=config.MODEL_NAME,
    num_classes=config.NUM_CLASSES,
    pretrained=True
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {config.MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel Architecture:")
print(model)

## 8. Define Loss Function, Optimizer, and Scheduler

In [ ]:
# Loss function (handles class imbalance)
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5,
    verbose=True,
    min_lr=1e-7
)

print("Optimizer, Loss function, and Scheduler initialized!")

## 9. Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train for one epoch.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels, _ in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': running_loss / (pbar.n + 1), 
                         'acc': correct / total})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """
    Validate the model.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validating')
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': running_loss / (pbar.n + 1), 
                             'acc': correct / total})
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels

print("Training and validation functions defined!")

## 10. Train the Model

In [ ]:
# Initialize history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
patience_counter = 0

# Training loop
for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch [{epoch+1}/{config.NUM_EPOCHS}]")
    print(f"{'='*60}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    # Validate
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Update learning rate
    scheduler.step(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), config.BEST_MODEL_PATH)
        print(f"✓ Best model saved! Val Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{config.PATIENCE}")
    
    # Early stopping
    if patience_counter >= config.PATIENCE:
        print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
        break

# Save final model
torch.save(model.state_dict(), config.FINAL_MODEL_PATH)
print(f"\n✓ Training completed! Models saved to {config.CHECKPOINT_DIR}")

## 11. Plot Training History

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.CHECKPOINT_DIR, 'training_history.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Training history plot saved!")

## 12. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(config.BEST_MODEL_PATH))

# Test
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Testing')
    for images, labels, _ in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Calculate metrics
test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"\n{'='*60}")
print(f"Test Results")
print(f"{'='*60}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1-Score (weighted): {test_f1:.4f}")
print(f"\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds, 
                           target_names=list(config.EMOTION_CLASSES.values())))

## 13. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=list(config.EMOTION_CLASSES.values()),
            yticklabels=list(config.EMOTION_CLASSES.values()),
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.savefig(os.path.join(config.CHECKPOINT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved!")

## 14. Inference on Images

In [ ]:
def predict_emotion(image_path, model, device, transform, emotion_classes):
    """
    Predict emotion for a single image.
    
    Args:
        image_path: Path to image file
        model: Trained model
        device: Device to use
        transform: Image transform
        emotion_classes: Dictionary of emotion classes
    
    Returns:
        emotion: Predicted emotion
        confidence: Confidence score
        all_probs: Probabilities for all classes
    """
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(image_tensor)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probs, 1)
    
    emotion = emotion_classes[predicted.item()]
    confidence = confidence.item()
    all_probs = probs.cpu().numpy()[0]
    
    return emotion, confidence, all_probs


def visualize_prediction(image_path, emotion, confidence, all_probs, emotion_classes):
    """
    Visualize prediction with probabilities.
    """
    image = Image.open(image_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Image
    axes[0].imshow(image)
    axes[0].set_title(f'Prediction: {emotion}\nConfidence: {confidence:.2%}')
    axes[0].axis('off')
    
    # Probabilities
    emotions = list(emotion_classes.values())
    colors = ['green' if e == emotion else 'gray' for e in emotions]
    axes[1].barh(emotions, all_probs, color=colors, alpha=0.7)
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Emotion Probabilities')
    axes[1].set_xlim([0, 1])
    
    # Add value labels
    for i, (emotion_name, prob) in enumerate(zip(emotions, all_probs)):
        axes[1].text(prob, i, f' {prob:.3f}', va='center')
    
    plt.tight_layout()
    plt.show()

print("Prediction functions defined!")

## 15. Test on Sample Images

In [ ]:
# Load best model
model.load_state_dict(torch.load(config.BEST_MODEL_PATH))

# Test on a few sample images from test set
print("Testing on sample images from test set...\n")

sample_count = 0
for images, labels, paths in test_loader:
    for image, label, path in zip(images, labels, paths):
        if sample_count >= 3:  # Show 3 samples
            break
        
        emotion, confidence, all_probs = predict_emotion(
            path, model, device, test_transform, config.EMOTION_CLASSES
        )
        
        true_emotion = config.EMOTION_CLASSES[label.item()]
        
        print(f"Image: {path}")
        print(f"True emotion: {true_emotion}")
        print(f"Predicted emotion: {emotion}")
        print(f"Confidence: {confidence:.4f}")
        print(f"Correct: {'✓' if emotion == true_emotion else '✗'}\n")
        
        visualize_prediction(path, emotion, confidence, all_probs, config.EMOTION_CLASSES)
        
        sample_count += 1
    
    if sample_count >= 3:
        break

## 16. Video Emotion Detection

In [ ]:
def detect_emotion_in_video(video_path, model, device, transform, emotion_classes, 
                            frame_interval=5, display=False):
    """
    Detect emotions in video frames.
    
    Args:
        video_path: Path to video file
        model: Trained model
        device: Device to use
        transform: Image transform
        emotion_classes: Dictionary of emotion classes
        frame_interval: Process every nth frame
        display: Whether to display frames with predictions
    
    Returns:
        results: List of predictions with timestamps
    """
    model.eval()
    cap = cv2.VideoCapture(video_path)
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = 0
    results = []
    
    emotion_counts = {emotion: 0 for emotion in emotion_classes.values()}
    
    with torch.no_grad():
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % frame_interval == 0:
                # Convert BGR to RGB
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image = Image.fromarray(rgb_frame)
                
                # Predict
                image_tensor = transform(image).unsqueeze(0).to(device)
                outputs = model(image_tensor)
                probs = torch.softmax(outputs, dim=1)
                confidence, predicted = torch.max(probs, 1)
                
                emotion = emotion_classes[predicted.item()]
                conf = confidence.item()
                timestamp = frame_count / fps
                
                results.append({
                    'frame': frame_count,
                    'timestamp': timestamp,
                    'emotion': emotion,
                    'confidence': conf
                })
                
                emotion_counts[emotion] += 1
                
                if display:
                    # Add text to frame
                    cv2.putText(frame, f'{emotion} ({conf:.2f})', (10, 30),
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    cv2.putText(frame, f'Frame: {frame_count}', (10, 70),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1)
                    
                    # Display frame (resize for notebook)
                    display_frame = cv2.resize(frame, (640, 480))
                    display_frame_rgb = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)
                    plt.imshow(display_frame_rgb)
                    plt.axis('off')
                    plt.show()
            
            frame_count += 1
    
    cap.release()
    
    # Summary statistics
    print(f"\nVideo Analysis Summary:")
    print(f"Total frames processed: {len(results)}")
    print(f"Emotion distribution:")
    for emotion, count in emotion_counts.items():
        if count > 0:
            print(f"  {emotion}: {count}")
    
    return results, emotion_counts

print("Video emotion detection function defined!")

## 17. Example: Process a Video (uncomment to use)

In [ ]:
# Example usage:
# video_path = './your_video.mp4'
# results, emotion_counts = detect_emotion_in_video(
#     video_path, model, device, test_transform, config.EMOTION_CLASSES,
#     frame_interval=5, display=False
# )

print("Uncomment above to process a video!")

## 18. Save Model for Production

In [ ]:
# Save model with metadata
checkpoint = {
    'model_state_dict': model.state_dict(),
    'model_name': config.MODEL_NAME,
    'num_classes': config.NUM_CLASSES,
    'emotion_classes': config.EMOTION_CLASSES,
    'img_size': config.IMG_SIZE,
    'test_accuracy': test_acc,
    'test_f1': test_f1,
}

torch.save(checkpoint, os.path.join(config.CHECKPOINT_DIR, 'emotion_model_checkpoint.pth'))

print(f"✓ Model checkpoint saved!")
print(f"Location: {os.path.join(config.CHECKPOINT_DIR, 'emotion_model_checkpoint.pth')}")
print(f"\nModel Performance Summary:")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Test F1-Score: {test_f1:.4f}")
print(f"  Best Val Accuracy: {best_val_acc:.4f}")

## 19. Load and Use Saved Model

In [ ]:
def load_emotion_model(checkpoint_path, device):
    """
    Load trained emotion detection model from checkpoint.
    """
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model_name = checkpoint['model_name']
    num_classes = checkpoint['num_classes']
    emotion_classes = checkpoint['emotion_classes']
    img_size = checkpoint['img_size']
    
    # Create model
    model = EmotionDetectionModel(
        model_name=model_name,
        num_classes=num_classes,
        pretrained=False
    ).to(device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model, emotion_classes, img_size


# Example: Load model for inference
# loaded_model, loaded_emotions, loaded_size = load_emotion_model(
#     './checkpoints/emotion_model_checkpoint.pth', device
# )

print("Model loading function defined!")

## 20. Summary and Next Steps

### ✓ Completed
1. **Data Loading**: Custom dataset class with support for organized emotion folders
2. **Preprocessing**: Image augmentation and normalization
3. **Model Architecture**: Fine-tuned ResNet50/EfficientNet with custom classification head
4. **Training Pipeline**: Full training loop with early stopping and best model saving
5. **Evaluation**: Comprehensive metrics (accuracy, F1, confusion matrix)
6. **Inference**: Single image and video emotion detection
7. **Visualization**: Training history, confusion matrix, prediction details

### 🚀 Next Steps
1. **Hyperparameter Tuning**: Experiment with learning rates, batch sizes, augmentation
2. **Ensemble Models**: Combine multiple models for better predictions
3. **Real-time Detection**: Deploy with camera input using OpenCV
4. **API Development**: Create REST API for model serving
5. **Further Optimization**: Quantization and pruning for faster inference

### 📊 Expected Performance
- **ResNet50**: 85-92% accuracy on balanced datasets
- **EfficientNet**: 87-94% accuracy with better efficiency
- **Training time**: 2-4 hours on GPU, 8-16 hours on CPU

Good luck with your emotion detection model! 🎉